# 07 - Model Export

Retrain the final **Logistic Regression Pipeline** (StandardScaler + LR) on the **full dataset** and save it for the API.

Feature set: the **6 features** retained by notebook 06 sequential backward elimination.


In [1]:
import pandas as pd
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv("../src/data/processed/engineered_dataset.csv")
df["long_lasting"] = (df["relationship_longevity_months"] >= 120).astype(int)

FEATURES = [
    "career_ambition_diff",
    "chronotype_diff",
    "emotional_expressiveness_diff",
    "openness_diff",
    "spontaneity_diff",
    "same_love_language",
]

X = df[FEATURES]
y = df["long_lasting"]

print(f"Training on full dataset : {len(X):,} rows, {X.shape[1]} features")
print(f"Features : {FEATURES}")
print(f"Positives: {y.sum():,} ({y.mean()*100:.2f}%)")


Training on full dataset : 100,000 rows, 6 features
Features : ['career_ambition_diff', 'chronotype_diff', 'emotional_expressiveness_diff', 'openness_diff', 'spontaneity_diff', 'same_love_language']
Positives: 940 (0.94%)


In [3]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])

pipeline.fit(X, y)
print("Training done.")


Training done.


In [4]:
os.makedirs("../src/models", exist_ok=True)

joblib.dump(pipeline, "../src/models/lr_pipeline_final.pkl")
print("Model saved to src/models/lr_pipeline_final.pkl")


Model saved to src/models/lr_pipeline_final.pkl


## Smoke test — reload & predict


In [5]:
loaded = joblib.load("../src/models/lr_pipeline_final.pkl")

sample = X.iloc[:3].copy()
probas = loaded.predict_proba(sample)[:, 1]

for i, p in enumerate(probas):
    print(f"Sample {i+1} — P(long-lasting) = {p:.4f}  →  "
          f"{'✓ durable (≥10y)' if p >= 0.5 else '✗ unlikely (<10y)'}")

print("\nSmoke test passed — pipeline loads and predicts correctly.")
print(f"Pipeline steps : {[s for s, _ in loaded.steps]}")
print(f"Features expected : {FEATURES}")


Sample 1 — P(long-lasting) = 0.1880  →  ✗ unlikely (<10y)
Sample 2 — P(long-lasting) = 0.6169  →  ✓ durable (≥10y)
Sample 3 — P(long-lasting) = 0.5767  →  ✓ durable (≥10y)

Smoke test passed — pipeline loads and predicts correctly.
Pipeline steps : ['scaler', 'lr']
Features expected : ['career_ambition_diff', 'chronotype_diff', 'emotional_expressiveness_diff', 'openness_diff', 'spontaneity_diff', 'same_love_language']
